# TF-IDF + SVM

**TF-IDF** (Term Frequency-Inverse Document Frequency) converts each document into a very long, sparse vector of numbers. Each position in the vector corresponds to a word (or word pair) from vocabulary. The number at that position is high if the word appears often in this document but rarely in other documents in the corpus. Words that appear everywhere ("the", "and", "I") get low weights; words that are distinctive to certain documents get high weights.
For cognitive distortion classification, this is a surprisingly strong signal because distortions often have characteristic vocabulary: "always", "never", "everyone" for overgeneralization; "should", "must", "ought" for should statements; "failure", "worthless" for labelling.

**SVM** (Support Vector Machine) is a classifier that, given these vectors, finds hyperplanes separating the categories. For multi-class problems, scikit-learn implements one-vs-rest internally. SVM handles high-dimensional sparse features well, which is exactly what TF-IDF produces.

- The `stratify` argument in `train_test_split` is important because the classes are imbalanced -> without stratification, some rare classes might end up only in the test set. The `class_weight='balanced'` argument tells SVM to weight minority classes more heavily during training, which partially compensates for the class imbalance. Without it, the model just learns to predict the majority class.

- The n-gram range of (1, 2) — unigrams and bigrams — matches what Shreevastava and Foltz used. Going up to trigrams sometimes helps but often doesn't, and adds parameters.

- `stop_words='english'` is debatable. Removing stopwords helps for topic classification but can hurt for cognitive distortions because words like "should", "must", "always", "never" are on many stopword lists but are exactly the signal needed. Both are tested.


**Prepare *TherapistQA* dataset**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load preprocessed TherapistQA from Stage 0
df = pd.read_csv('../../processed/therapistqa.csv')

# Split - use the same split every time for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    df['patient_question'],
    df['dominant_distortion'],
    test_size=0.2,
    stratify=df['dominant_distortion'],
    random_state=42
)

**Prepare *PatternReframe* dataset**

In [1]:
import pandas as pd

# Load PatternReframe splits from Stage 0 preprocessing
df_train = pd.read_csv('../../processed/patternreframe_train.csv')
df_test = pd.read_csv('../../processed/patternreframe_test.csv')

X_train = df_train['thought']
y_train = df_train['pattern']
X_test = df_test['thought']
y_test = df_test['pattern']

print(f"PatternReframe train: {len(X_train)} records")
print(f"PatternReframe test: {len(X_test)} records")
print(f"Train label distribution:")
print(y_train.value_counts())
print(f"\nTest label distribution:")
print(y_test.value_counts())

PatternReframe train: 1920 records
PatternReframe test: 6807 records
Train label distribution:
pattern
Catastrophizing                                                    200
Jumping to conclusions: Fortune-telling                            199
Overgeneralization                                                 197
Jumping to conclusions: mind reading                               197
Labeling and mislabeling                                           196
Discounting the positive                                           194
Black-and-white or polarized thinking / All or nothing thinking    188
Personalization                                                    185
Mental filtering                                                   185
Should statements                                                  179
Name: count, dtype: int64

Test label distribution:
pattern
Catastrophizing                                                    726
Jumping to conclusions: Fortune-telling                 

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),            # unigrams and bigrams
        min_df=2,                      # ignore words appearing in <2 documents
        max_df=0.95,                   # ignore words appearing in >95% of documents
        sublinear_tf=True,             # dampen very frequent terms
        stop_words='english'           # remove English stopwords
    )),
    ('svm', LinearSVC(
        C=1.0,                         # regularisation strength
        class_weight='balanced',       # important given class imbalance
        max_iter=5000
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

                                                                 precision    recall  f1-score   support

Black-and-white or polarized thinking / All or nothing thinking      0.250     0.251     0.251       665
                                                Catastrophizing      0.439     0.424     0.431       726
                                       Discounting the positive      0.376     0.401     0.388       681
                        Jumping to conclusions: Fortune-telling      0.253     0.238     0.245       711
                           Jumping to conclusions: mind reading      0.441     0.561     0.494       699
                                       Labeling and mislabeling      0.360     0.324     0.341       669
                                               Mental filtering      0.279     0.236     0.256       660
                                             Overgeneralization      0.275     0.276     0.275       688
                                                Person

### Binary detection number
The 11-way problem is reduced to 'distorted vs non-distorted'

In [3]:
import numpy as np

# Collapse to binary
y_test_binary = np.where(y_test == 'No Distortion', 'No Distortion', 'Distorted')
y_pred_binary = np.where(y_pred == 'No Distortion', 'No Distortion', 'Distorted')

print(classification_report(y_test_binary, y_pred_binary, digits=3))

              precision    recall  f1-score   support

   Distorted      1.000     1.000     1.000      6807

    accuracy                          1.000      6807
   macro avg      1.000     1.000     1.000      6807
weighted avg      1.000     1.000     1.000      6807



### Ablation: stopword removal
Keep stopwords.

In [4]:
# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),            # unigrams and bigrams
        min_df=2,                      # ignore words appearing in <2 documents
        max_df=0.95,                   # ignore words appearing in >95% of documents
        sublinear_tf=True,             # dampen very frequent terms
    )),
    ('svm', LinearSVC(
        C=1.0,                         # regularisation strength
        class_weight='balanced',       # important given class imbalance
        max_iter=5000
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

                                                                 precision    recall  f1-score   support

Black-and-white or polarized thinking / All or nothing thinking      0.420     0.308     0.356       665
                                                Catastrophizing      0.457     0.488     0.472       726
                                       Discounting the positive      0.585     0.598     0.591       681
                        Jumping to conclusions: Fortune-telling      0.324     0.307     0.315       711
                           Jumping to conclusions: mind reading      0.545     0.638     0.588       699
                                       Labeling and mislabeling      0.511     0.442     0.474       669
                                               Mental filtering      0.443     0.453     0.448       660
                                             Overgeneralization      0.390     0.436     0.412       688
                                                Person

### Ablation: n-gram range
Use trigrams with unigrams

In [5]:
# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),            # unigrams and trigrams
        min_df=2,                      # ignore words appearing in <2 documents
        max_df=0.95,                   # ignore words appearing in >95% of documents
        sublinear_tf=True,             # dampen very frequent terms
        stop_words='english'           # remove English stopwords
    )),
    ('svm', LinearSVC(
        C=1.0,                         # regularisation strength
        class_weight='balanced',       # important given class imbalance
        max_iter=5000
    ))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

                                                                 precision    recall  f1-score   support

Black-and-white or polarized thinking / All or nothing thinking      0.251     0.247     0.249       665
                                                Catastrophizing      0.444     0.435     0.439       726
                                       Discounting the positive      0.381     0.408     0.394       681
                        Jumping to conclusions: Fortune-telling      0.253     0.235     0.244       711
                           Jumping to conclusions: mind reading      0.442     0.564     0.496       699
                                       Labeling and mislabeling      0.357     0.320     0.337       669
                                               Mental filtering      0.279     0.239     0.258       660
                                             Overgeneralization      0.281     0.281     0.281       688
                                                Person